In [1]:
# %matplotlib widget
import py4DSTEM 
import numpy as np
import h5py as h5
import matplotlib.pyplot as plt
import hyperspy.api as hs
import os, json
from matplotlib.animation import FuncAnimation
print(py4DSTEM.__file__)
print(py4DSTEM.__version__)

/dls_sw/apps/python/miniforge/25.11/envs/epsic3.13/lib/python3.13/site-packages/py4DSTEM/__init__.py
0.14.19


data_path=
skip_real=2
bin_q=2
thresh_upper=0.1
save_path=
def_start=-300
def_end=300
def_step=25

In [ ]:
# data_path='/dls/e01/data/2025/cm40602-5/raw/Ptycho_Pt-C_session-day2/SI-001/data.hdf5'
skip_real=int(skip_real)
bin_q=int(bin_q)
thresh_upper=float(thresh_upper)
# save_path='/dls/e01/data/2025/cm40602-5/processing/array_analysis'
def_start=int(def_start)
def_end=int(def_end)
def_step=int(def_step)

# Take HT from metadata
# f['metadata/ht_value(V)'][()]

In [ ]:
fov_4mx = 5.0000000000000004e-08 # in m
step_size = fov_4mx / 256 # in m for 256 ROI datasets
print(step_size)
probe_conv_semi = 0.0534 / 2 # in rad using 30 um CL ap
print(probe_conv_semi)

In [ ]:
data_name = data_path.split('/')[-2]
save_path_folder = os.path.join(os.path.dirname(data_path), save_path)
print(save_path_folder)
if not os.path.exists(save_path_folder):
    os.mkdir(save_path_folder)

In [ ]:
with h5.File(data_path) as f:
    d =  f['Experiments/__unnamed__/data'][::skip_real,::skip_real,90:460, 80:450][()]
    # [::skip_real,::skip_real,120:490, 80:450][()]

mask_path = '/dls/science/groups/e02/Frederick/DetecotrMasks/29042024_12bitmask2.h5'
# '/dls_sw/e02/medipix_mask/29042024_12bitmask2.h5'
with h5.File(mask_path, 'r') as f:
    mask = f['data/mask'][()]
mask_binary = mask > 0.5

In [ ]:
pwd

In [ ]:
masked_data = d.data * np.invert(mask_binary[90:460, 80:450])

In [ ]:
datacube = py4DSTEM.datacube.DataCube(data = masked_data)

In [ ]:
datacube = datacube.bin_Q(bin_q)

In [ ]:
datacube.get_dp_mean();

py4DSTEM.show(
    [
        datacube.tree('dp_mean'),
    ]
)

In [ ]:
probe_radius_pixels, probe_qx0, probe_qy0 = datacube.get_probe_size(
    datacube.data[0,0,:,:],
    plot = True,
    thresh_upper = 0.1
)

In [ ]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path_folder, f'{data_name}_BF_disc_detection_v2.PNG'))

In [ ]:
datacube.calibration.set_R_pixel_size(step_size * 1e10 * skip_real)
datacube.calibration.set_R_pixel_units('A')
datacube.calibration.set_Q_pixel_size(bin_q * probe_conv_semi * 1e3 / probe_radius_pixels)
datacube.calibration.set_Q_pixel_units('mrad')
datacube.calibration

In [ ]:
expand_BF = 2.0

center = (probe_qx0, probe_qy0)
radius_BF = probe_radius_pixels + expand_BF
radii_DF = (probe_radius_pixels + expand_BF, 1e3)

datacube.get_virtual_image(
    mode = 'circle',
    geometry = (center,radius_BF),
    name = 'bright_field',
    shift_center = False,
)
datacube.get_virtual_image(
    mode = 'annulus',
    geometry = (center,radii_DF),
    name = 'dark_field',
    shift_center = False,
);

fig, ax = py4DSTEM.show(
    [
        datacube.tree('bright_field'),
        datacube.tree('dark_field'),               
    ],
    cmap='viridis',
    ticks = False,
    axsize = (4,4),
    title = ['Bright Field','Dark Field'],
    returnfig=True,
)

In [ ]:
fig.savefig(os.path.join(save_path_folder, f'{data_name}_BF_DF_images_v2.PNG'))

In [ ]:
# with h5.File(fp) as f:  #if necessary crop the dataset in this step!
     # idex part of an array with: f['/data/frames'][0:24, 0:22, 0:123, 0:234][()]
    # d = f['/data/frames'][()]

In [ ]:
parallax = py4DSTEM.process.phase.Parallax(
    datacube=datacube,
    energy = 300e3,
    device = "gpu", #"cpu", 
    verbose = True,
    object_padding_px=(4,4),
).preprocess(
    normalize_images=False,
    plot_average_bf=True,
    threshold_intensity=0.6,
    edge_blend=2,
)

In [ ]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path_folder, f'{data_name}_parallax_preprocess_v2.PNG'))

In [ ]:
parallax = parallax.reconstruct(
    reset=True,
    min_alignment_bin=32,
    num_iter_at_min_bin = 6,
)

In [ ]:
parallax.show_shifts(
    plot_arrow_freq=4
)

In [ ]:
parallax.subpixel_alignment(
    kde_upsample_factor=2,
    plot_upsampled_BF_comparison=True,
    plot_upsampled_FFT_comparison=True,
)

In [ ]:
parallax = parallax.aberration_fit(
    max_radial_order = 3,
    plot_CTF_comparison = True,
)

In [ ]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path_folder, f'{data_name}_CTF_fit_v2.PNG'))

In [ ]:
aberr_dict = parallax.aberrations_dict_cartesian
with open(os.path.join(save_path_folder, f'{data_name}_aberration_fit_v2.json'), 'w') as f:
    json.dump(aberr_dict, f, indent=4)

In [ ]:
parallax = parallax.aberration_correct(
    use_CTF_fit=True,
)

In [ ]:
fig = plt.gcf()
fig.savefig(os.path.join(save_path_folder, f'{data_name}_CTF_corrected_v2.PNG'))

In [ ]:
defocus_list = np.arange(def_start, def_end, def_step) #in Ang!
print(defocus_list)
defocus_arrays = []
for this_defocus in defocus_list:
    print(f'calculating {this_defocus} Ang')
    defocus_arrays.append(parallax.depth_section(
                                    depth_angstroms= this_defocus,
                                    use_CTF_fit= False) # true/false?? try both
                         )


In [ ]:
# Assuming we are running on GPU
defocus_arrays = np.array([item.get() for item in defocus_arrays]) # np.array(defocus_arrays.get())
defocus_arrays = np.squeeze(defocus_arrays)

In [ ]:
defocus_arrays.shape

In [ ]:
defocus_stack = hs.signals.Signal2D(defocus_arrays)
defocus_stack.axes_manager[0].scale = def_step / 10
defocus_stack.axes_manager[0].units = 'nm'
defocus_stack.axes_manager[0].offset = def_start / 10


In [ ]:
# defocus_stack.plot()

In [ ]:
defocus_stack.save(os.path.join(save_path_folder, f'{data_name}_def_sweep_stack_v2.hspy'), overwrite=True)

In [ ]:
nav_axis = defocus_stack.axes_manager.navigation_axes[0]
units = nav_axis.units
scale = nav_axis.scale
offset = nav_axis.offset

fig, ax = plt.subplots()
im = ax.imshow(defocus_stack.inav[0].data, cmap='viridis')

def update(frame):
    # Calculate the calibrated value for the current frame
    calibrated_value = (frame * scale) + offset
    
    im.set_data(defocus_stack.inav[frame].data)
    
    # Update title with units (e.g., "Position: 10.5 nm")
    ax.set_title(f"Defocus value: {calibrated_value:.2f} {units}")
    
    return [im]

ani = FuncAnimation(fig, update, frames=defocus_stack.axes_manager.navigation_shape[0], 
                    interval=200, blit=True)

ani.save(os.path.join(save_path_folder, f'{data_name}_def_sweep_stack_v2.gif'), writer='pillow')